# ResNet experts (Pool B)

Trains **one ResNet per expert** (real vs. that expert's fakes), calibrates it, and scores every image the
combiners use with every expert: the ResNet half of the score table. See `resnet/train.py` and `resnet/score.py`.

- **7 experts**, the same as the CLIP pool: ADM, BigGAN, glide, Midjourney, SD (SD 1.4 + 1.5 together), VQDM, wukong (`resnet/experts.py`).
- **Train:** ImageNet-pretrained ResNet, fine-tuned on the expert's own `expert_train` images (random 224 crop + flip).
  `expert_val` picks the best epoch (lowest loss, early stopping) and fits the temperature. No other pool is touched.
- **Score:** every image of `expert_val`, `meta_train`, `meta_val`, `test` (and any pool added later, e.g. a seen-test pool)
  → one raw logit per expert, saved as the feature set `resnet50_experts`. `load_scores()` returns calibrated
  probabilities `sigmoid(logit / T)`.

**Time on an M4 (16 GB), ResNet-50:** ~30 min per expert (≤ 8 epochs of 10k images at ~42 images/s), so ~2.5-3.5 h
for all 7, plus ~1.2 h scoring. `ARCH = "resnet18"` is ~2.5x faster.

**Safe to stop and rerun:** finished experts and scored images are saved and skipped.
Set `LIMIT = 100` first for a test run of the whole notebook (~10 min); it saves to separate folders.

In [ ]:
import sys
sys.path.insert(0, "../..")  # repo root, so `common`, `data` and `resnet` are importable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from data import config
from resnet.experts import EXPERTS
from resnet.score import check, cross_auroc, load_scores, score
from resnet.train import train_all

ARCH = "resnet50"  # "resnet18" if time is short
LIMIT = 100  # e.g. 100 for a test run: 100 images per label and pool, saved separately
EPOCHS = 1 if LIMIT else 8
BATCH_SIZE = 64  # training; lower to 32 if you run out of memory
NUM_WORKERS = 4

print("dataset:", config.dataset_dir(), config.dataset_dir().exists())

## 1 · Train and calibrate the 7 experts

In [ ]:
infos = train_all(epochs=EPOCHS, limit=LIMIT, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, arch=ARCH)

### Training curves

Validation loss is on each expert's own `expert_val` images. `T` is the calibration temperature:
1 means the expert was already calibrated (or perfectly separated on validation), larger values soften it.

In [ ]:
# Categorical slots in fixed order, one per expert.
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6"]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharex=True)
for color, (expert, info) in zip(COLORS, infos.items()):
    history = pd.DataFrame(info["history"])
    axes[0].plot(history.epoch, history.train_loss, color=color, lw=2, marker="o", ms=4, label=expert)
    axes[1].plot(history.epoch, history.val_loss, color=color, lw=2, marker="o", ms=4, label=expert)
for ax, title in zip(axes, ["Training loss", "Validation loss (expert_val)"]):
    ax.set_title(title, loc="left")
    ax.set_xlabel("epoch")
    ax.set_ylabel("BCE loss")
    ax.grid(alpha=0.25)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.spines[["top", "right"]].set_visible(False)
axes[1].legend(frameon=False, fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

pd.DataFrame([{
    "expert": expert, "best epoch": info["best_epoch"], "epochs run": info["epochs_run"],
    "val auc": info["val"]["auc"], "val acc": info["val"]["acc"], "val loss": info["val"]["loss"],
    "calibrated loss": info["val_calibrated_loss"], "T": info["temperature"],
    "minutes": info["train_seconds"] / 60,
} for expert, info in infos.items()]).round(4)

## 2 · Score every image with every expert

In [ ]:
score(limit=LIMIT, num_workers=NUM_WORKERS, arch=ARCH)
missing = check(limit=LIMIT, arch=ARCH)

## 3 · Sanity check: who detects whom?

AUROC of every expert (rows) on every generator's `meta_val` images (columns). `test` is not looked at.

- **Diagonal ≈ 1.0:** each expert detects its own generator(s).
- **Siblings high:** SD ↔ wukong, ADM → glide / BigGAN (why those are held out together / labelled "easy").
- **Most other cells much lower:** ResNet experts are specialists.

A low diagonal means training went wrong. *Everything* ≈ 1.0 points to a shortcut (image size or JPEG traces).

In [ ]:
table = cross_auroc("meta_val", limit=LIMIT, arch=ARCH)
short = {"stable_diffusion_v_1_4": "SD 1.4", "stable_diffusion_v_1_5": "SD 1.5"}
values = table.to_numpy(dtype=float)

fig, ax = plt.subplots(figsize=(7.6, 5.4))
cmap = plt.matplotlib.colors.LinearSegmentedColormap.from_list(
    "blue", ["#f0efec", "#9ec5f4", "#3987e5", "#1c5cab", "#0d366b"])  # one hue, light -> dark
image = ax.imshow(values, cmap=cmap, vmin=0.5, vmax=1.0)
ax.set_xticks(range(len(table.columns)), [short.get(g, g) for g in table.columns], rotation=35, ha="right")
ax.set_yticks(range(len(table.index)), table.index)
ax.set_xlabel("images from generator (meta_val)")
ax.set_ylabel("expert")
ax.set_title(f"ResNet experts: AUROC per generator ({ARCH})", loc="left")
for i in range(values.shape[0]):
    for j in range(values.shape[1]):
        if not np.isnan(values[i, j]):
            ax.text(j, i, f"{values[i, j]:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if values[i, j] > 0.8 else "#0b0b0b")
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04, label="AUROC (0.5 = chance)")
plt.tight_layout()
plt.show()
table.round(3)

## 4 · Using the scores in the combiners

Same shape as `data.features.load_features`: `X` has one column per expert in `EXPERTS` order,
`rows` the matching manifest rows. Drop the held-out expert's column per fold.

In [ ]:
X, rows = load_scores(pool="meta_train", limit=LIMIT, arch=ARCH)  # calibrated=False for raw logits
print(X.shape, list(EXPERTS))
pd.DataFrame(X, columns=list(EXPERTS)).assign(generator=rows.generator.values, label=rows.label.values).head()